# Evaluate Rewrite Quality on Multi-turn Scenarios

Đánh giá chất lượng module viết lại của router Qwen3-4B FT v7.1 trên bộ 100 hội thoại / 308 lượt.

**Chỉ chạy router**, không gọi agent. Đo 4 metric per-turn:
1. **Intent accuracy** — router phân loại đúng ý định
2. **Scope accuracy** — router chọn đúng cấp địa lý (city/district/ward)
3. **Location resolution accuracy** — rewrite có chứa đúng địa danh kỳ vọng
4. **Rewrite quality (G-Eval)** — LLM judge 2 chiều:
   - Faithfulness: rewrite giữ đúng ngữ cảnh các lượt trước
   - Self-containedness: rewrite đủ thông tin để hiểu độc lập

**Output**: `outputs/summary_rewrite_quality.json` + `outputs/raw_turns.jsonl`

**Prerequisites**:
- Upload `multi_turn_scenarios.jsonl` và `system_prompt.txt` lên Colab
- Ollama đã chạy với model `hanoi-weather-router` (notebook này hoặc notebook khác)
- `JUDGE_API_BASE` + `JUDGE_API_KEY` (OpenAI-compatible endpoint, khớp `experiments/evaluation/configs/judge.yaml`)


## 1. Setup dependencies

In [ ]:
!pip install -q httpx pandas tqdm pydantic openai nest_asyncio huggingface_hub


## 2. Config flags

Sửa các flag dưới đây cho phù hợp với môi trường Colab của bạn:
- `AUTO_SETUP_OLLAMA = True` nếu muốn notebook tự cài Ollama + pull model. Nếu Ollama đã chạy ở notebook khác → giữ `False`.
- `OLLAMA_BASE_URL` mặc định localhost — sửa nếu Ollama chạy qua tunnel/remote.
- `OLLAMA_MODEL` là tên model trên Ollama (default `hanoi-weather-router`).
- `JUDGE_API_BASE` + `JUDGE_API_KEY`: endpoint OpenAI-compatible cho judge (match laptop `experiments/evaluation/configs/judge.yaml`).
- `JUDGE_MODEL` default `gpt-4o` (khớp laptop).


In [ ]:
import os

AUTO_SETUP_OLLAMA = True  # True: notebook tự install + start Ollama
SCENARIOS_PATH = 'multi_turn_scenarios.jsonl'
SYSTEM_PROMPT_PATH = 'system_prompt.txt'
OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL', 'hanoi-weather-router')
OUTPUT_PATH = 'outputs/summary_rewrite_quality.json'
RAW_TURNS_PATH = 'outputs/raw_turns.jsonl'
JUDGE_API_BASE = 'https://gpt1.shupremium.com/v1'  # OpenAI-compat endpoint URL
JUDGE_API_KEY = 'sk-XPvfRds1xszrG4ciSEL4LTNkSdLTO2W6GQw0ijCDSNS9TADa'
JUDGE_MODEL = os.environ.get('JUDGE_MODEL', 'gpt-4o')
JUDGE_CONCURRENCY = 50
K_HISTORY = 3  # ChatML sliding window

os.makedirs('outputs', exist_ok=True)
print('OLLAMA_BASE_URL =', OLLAMA_BASE_URL)
print('OLLAMA_MODEL =', OLLAMA_MODEL)
print('JUDGE_API_BASE =', JUDGE_API_BASE or '(not set — paste in cell 5)')
print('JUDGE_MODEL =', JUDGE_MODEL)


OLLAMA_BASE_URL = http://localhost:11434
OLLAMA_MODEL = hanoi-weather-router
JUDGE_API_BASE = https://gpt1.shupremium.com/v1
JUDGE_MODEL = gpt-4o


## 3. Auto-setup Ollama + create `hanoi-weather-router` model

Khớp đúng cấu hình production ở [scripts/colab/ollama_router_colab.ipynb](scripts/colab/ollama_router_colab.ipynb):
- Install Ollama
- Start serve trên `0.0.0.0:11434`
- Download GGUF `daredevil467/hanoi-router-qwen3-4b-v7-1-gguf` từ HuggingFace
- Tạo Modelfile với system_prompt v7.1 + sampling params (T=0.6, top_p=0.95, ...)
- `ollama create hanoi-weather-router`

Nếu đã có Ollama chạy ở notebook khác (qua tunnel) → set `AUTO_SETUP_OLLAMA = False` ở cell config và override `OLLAMA_BASE_URL`.


In [ ]:
if AUTO_SETUP_OLLAMA:
    import subprocess, time

    # 3.1 Install Ollama
    !apt-get -qq install -y zstd
    !curl -fsSL https://ollama.com/install.sh | sh
    !/usr/local/bin/ollama --version

    # 3.2 Start serve
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    if '/usr/local/bin' not in os.environ['PATH']:
        os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=os.environ.copy(),
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

    # 3.3 Download system prompt v7.1
    from huggingface_hub import hf_hub_download
    prompt_path = hf_hub_download(
        repo_id='daredevil467/hanoi-weather-router-data-v7',
        filename='system_prompt_v7_1.txt',
        repo_type='dataset',
    )
    SYSTEM_PROMPT_V71 = open(prompt_path, encoding='utf-8').read()
    # Save local for cell 6 to reload
    with open(SYSTEM_PROMPT_PATH, 'w', encoding='utf-8') as f:
        f.write(SYSTEM_PROMPT_V71)
    print(f'System prompt v7.1: {len(SYSTEM_PROMPT_V71)} chars')

    # 3.4 Create Modelfile
    modelfile = f'''FROM hf.co/daredevil467/hanoi-router-qwen3-4b-v7-1-gguf

TEMPLATE """<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER top_k 20
PARAMETER min_p 0
PARAMETER repeat_penalty 1.1
PARAMETER num_predict 1024

SYSTEM """{SYSTEM_PROMPT_V71}"""
'''
    open('/content/Modelfile', 'w', encoding='utf-8').write(modelfile)

    # 3.5 Create model
    !ollama create hanoi-weather-router -f /content/Modelfile
    !ollama list
else:
    print('AUTO_SETUP_OLLAMA=False — assume Ollama is running externally.')
    print(f'OLLAMA_BASE_URL = {OLLAMA_BASE_URL}')


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
System prompt v7.1: 4551 chars

NAME                                                         ID              SIZE      MODIFIED               
hanoi-weather-router:latest                                  cfbe70f60833    2.5 GB    Less than a second ago    
hf.co/daredevil467/hanoi-router-qwen3-4b-v7-1-gguf:latest    31071c4aa527    2.5 GB    11 minutes ago            


## 4. Smoke test Ollama router

In [ ]:
import httpx, json

def smoke_test():
    payload = {
        'model': OLLAMA_MODEL,
        'messages': [
            {'role': 'system', 'content': 'Test'},
            {'role': 'user', 'content': 'Trời Hà Nội thế nào?'},
        ],
        'stream': False,
        'options': {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'num_predict': 256},
    }
    r = httpx.post(f'{OLLAMA_BASE_URL}/api/chat', json=payload, timeout=60)
    r.raise_for_status()
    body = r.json()
    content = body.get('message', {}).get('content', '')
    print('Smoke test OK. Sample response (first 300 chars):')
    print(content[:300])

smoke_test()


Smoke test OK. Sample response (first 300 chars):
{"status": "success", "data": {"city": "Hà Nội", "weather": "nhiệt độ 25°C, trời nắng nhẹ, không có mưa"}}


## 5. Set Judge API credentials (OpenAI-compatible)

Cần `JUDGE_API_BASE` (endpoint URL) + `JUDGE_API_KEY`. Khớp với cấu hình laptop `experiments/evaluation/configs/judge.yaml` (sv1 qwen-api hosting `gpt-4o`).

Nếu đã set env var từ trước thì cell này skip.


In [ ]:
import getpass

if not JUDGE_API_BASE:
    JUDGE_API_BASE = input('Paste JUDGE_API_BASE (e.g., https://sv1.example.com/v1): ').strip()
    os.environ['JUDGE_API_BASE'] = JUDGE_API_BASE

if not JUDGE_API_KEY:
    JUDGE_API_KEY = getpass.getpass('Paste JUDGE_API_KEY: ').strip()
    os.environ['JUDGE_API_KEY'] = JUDGE_API_KEY

assert JUDGE_API_BASE and JUDGE_API_KEY, 'Cần cả JUDGE_API_BASE và JUDGE_API_KEY'
print(f'Judge: model={JUDGE_MODEL}, base={JUDGE_API_BASE[:50]}...')


Judge: model=gpt-4o, base=https://gpt1.shupremium.com/v1...


## 6. Load scenarios + system prompt

In [ ]:
with open(SCENARIOS_PATH, 'r', encoding='utf-8') as f:
    conversations = [json.loads(line) for line in f if line.strip()]

with open(SYSTEM_PROMPT_PATH, 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()

total_turns = sum(len(c['turns']) for c in conversations)
patterns = {}
for c in conversations:
    patterns[c['pattern']] = patterns.get(c['pattern'], 0) + 1

print(f'Loaded {len(conversations)} conversations, {total_turns} turns')
print(f'Patterns: {patterns}')
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')


Loaded 100 conversations, 308 turns
Patterns: {'location_carry_over': 16, 'time_shift': 15, 'topic_drilldown': 15, 'location_switch': 15, 'anaphora_chain': 15, 'mixed_intent': 15, 'negation': 9}
System prompt: 4551 chars


## 7. Router client (production-parity)

Build ChatML messages với K=3 sliding window history. Strip `<think>...</think>` tags (Qwen3 thinking mode). Parse JSON 4-keys response.


In [ ]:
import re

def parse_router_response(raw):
    # Production-parity parsing: strip <think>, try json.loads, fallback regex
    text = raw.strip()
    text = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL).strip()
    null_result = {'intent': None, 'scope': None, 'confidence': 0.0, 'rewritten_query': None, '_raw': text}
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Fallback: first JSON object (1 level nested)
    match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            null_result['_parse_error'] = 'regex_match_invalid_json'
            return null_result
    null_result['_parse_error'] = 'no_json_found'
    return null_result

def build_messages(history, user_query):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for user_msg, asst_json in history[-K_HISTORY:]:
        msgs.append({'role': 'user', 'content': user_msg})
        msgs.append({'role': 'assistant', 'content': asst_json})
    msgs.append({'role': 'user', 'content': user_query})
    return msgs

def call_router(history, user_query):
    messages = build_messages(history, user_query)
    payload = {
        'model': OLLAMA_MODEL,
        'messages': messages,
        'stream': False,
        'keep_alive': '15m',
        'options': {
            'temperature': 0.6,
            'top_p': 0.95,
            'top_k': 20,
            'min_p': 0,
            'repeat_penalty': 1.1,
            'num_predict': 1024,
        },
    }
    r = httpx.post(f'{OLLAMA_BASE_URL}/api/chat', json=payload, timeout=180)
    r.raise_for_status()
    raw = r.json().get('message', {}).get('content', '')
    return parse_router_response(raw), raw


## 8. Loop eval — gọi router cho từng turn


In [ ]:
from tqdm.auto import tqdm

raw_turn_results = []
for conv in tqdm(conversations, desc='Conversations'):
    history = []
    for turn in conv['turns']:
        user_query = turn['user']
        try:
            result, raw = call_router(history, user_query)
            error = None
        except Exception as e:
            result, raw = {'intent': None, 'scope': None, 'confidence': 0.0, 'rewritten_query': None}, ''
            error = str(e)[:200]

        asst_json = json.dumps({
            'intent': result.get('intent'),
            'scope': result.get('scope'),
            'confidence': result.get('confidence'),
            'rewritten_query': result.get('rewritten_query'),
        }, ensure_ascii=False)

        raw_turn_results.append({
            'conversation_id': conv['conversation_id'],
            'pattern': conv['pattern'],
            'difficulty': conv.get('difficulty'),
            'turn': turn['turn'],
            'user': user_query,
            'expected_intent': turn.get('expected_intent'),
            'expected_scope': turn.get('expected_scope'),
            'expected_location': turn.get('expected_location'),
            'predicted_intent': result.get('intent'),
            'predicted_scope': result.get('scope'),
            'predicted_rewrite': result.get('rewritten_query'),
            'predicted_confidence': result.get('confidence'),
            'context_history_snapshot': list(history),
            'raw_truncated': raw[:500] if raw else '',
            'error': error,
        })
        history.append((user_query, asst_json))

with open(RAW_TURNS_PATH, 'w', encoding='utf-8') as f:
    for t in raw_turn_results:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')

n_ok = sum(1 for t in raw_turn_results if t['predicted_intent'])
print(f'Collected {len(raw_turn_results)} turn results ({n_ok} with predicted intent)')


Conversations:   0%|          | 0/100 [00:00<?, ?it/s]

Collected 308 turn results (308 with predicted intent)


## 9. G-Eval async — Faithfulness + Self-containedness

Chỉ chấm cho turn có `predicted_rewrite` non-null. Mỗi turn gọi 2 prompt (Faithfulness + Self-containedness). Async với semaphore=50 để limit concurrency.


In [ ]:
import asyncio
from openai import AsyncOpenAI
import nest_asyncio
nest_asyncio.apply()

JUDGE_FAITHFULNESS = '''Bạn là chuyên gia đánh giá câu viết lại trong hệ thống chatbot đa lượt.

Nhiệm vụ: chấm điểm Faithfulness của câu viết lại — mức độ rewrite giữ đúng ý định và ngữ cảnh từ các lượt trước.

Hội thoại lịch sử:
{history}

Câu hỏi hiện tại của user (có thể là fragment hoặc dùng đại từ):
{current}

Câu viết lại do router sinh ra:
{rewrite}

Tiêu chí Faithfulness (1-5):
- 1: Sai hoàn toàn — bóp méo ý định hoặc thay đổi ngữ cảnh
- 2: Có lỗi lớn — location/time/intent bị sai
- 3: Chấp nhận được — giữ ý định chính nhưng thiếu chi tiết
- 4: Tốt — giữ đúng ý định và đa số ngữ cảnh
- 5: Xuất sắc — giữ chính xác toàn bộ ý định và ngữ cảnh

Trả về JSON object: {{"reasoning": "giải thích ngắn", "score": N}} với N là số nguyên 1-5.'''

JUDGE_SELF_CONTAINED = '''Bạn là chuyên gia đánh giá câu viết lại trong hệ thống chatbot đa lượt.

Nhiệm vụ: chấm điểm Self-containedness — mức độ câu viết lại đủ thông tin để hiểu mà không cần lịch sử hội thoại.

Câu hỏi gốc của user (có thể là fragment hoặc dùng đại từ):
{current}

Câu viết lại do router sinh ra:
{rewrite}

Tiêu chí Self-containedness (1-5):
- 1: Vẫn cần ngữ cảnh để hiểu (vd vẫn còn đại từ chưa giải quyết)
- 2: Hơi tự chứa, thiếu nhiều chi tiết quan trọng (location, time)
- 3: Tự chứa cơ bản — có đủ thông tin chính
- 4: Tự chứa tốt — đầy đủ ý định, location, time
- 5: Hoàn toàn tự chứa — đọc độc lập vẫn rõ nghĩa

Trả về JSON object: {{"reasoning": "giải thích ngắn", "score": N}} với N là số nguyên 1-5.'''

async def call_judge(client, prompt, sem):
    async with sem:
        try:
            resp = await client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                response_format={'type': 'json_object'},
                temperature=0.0,
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            return {'reasoning': f'judge_error: {e}', 'score': None}

async def run_geval(turn_results):
    client = AsyncOpenAI(base_url=JUDGE_API_BASE, api_key=JUDGE_API_KEY)
    sem = asyncio.Semaphore(JUDGE_CONCURRENCY)

    tasks = []
    eligible_idx = []
    for i, t in enumerate(turn_results):
        rewrite = t.get('predicted_rewrite')
        if not rewrite:
            continue
        history_text = '\n'.join([f"User: {h[0]}\nAssistant: {h[1]}" for h in t['context_history_snapshot']]) or '(không có lịch sử — turn đầu tiên)'
        fait = JUDGE_FAITHFULNESS.format(history=history_text, current=t['user'], rewrite=rewrite)
        sc = JUDGE_SELF_CONTAINED.format(current=t['user'], rewrite=rewrite)
        tasks.append(call_judge(client, fait, sem))
        tasks.append(call_judge(client, sc, sem))
        eligible_idx.append(i)

    print(f'Running {len(tasks)} G-Eval calls (sem={JUDGE_CONCURRENCY})...')
    results = await asyncio.gather(*tasks)

    for idx, i in enumerate(eligible_idx):
        turn_results[i]['judge_faithfulness'] = results[idx*2]
        turn_results[i]['judge_self_contained'] = results[idx*2+1]

    return turn_results

raw_turn_results = asyncio.run(run_geval(raw_turn_results))

# Rewrite raw_turns with judge scores
with open(RAW_TURNS_PATH, 'w', encoding='utf-8') as f:
    for t in raw_turn_results:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')

n_judged = sum(1 for t in raw_turn_results if t.get('judge_faithfulness', {}).get('score') is not None)
print(f'G-Eval done. {n_judged} turns scored.')


Running 494 G-Eval calls (sem=50)...
G-Eval done. 0 turns scored.


## 10. Aggregate metrics theo pattern

Bảng kết quả khớp Bảng 4.x trong KLTN (7 pattern + overall, đơn vị %):
- `intent_accuracy` + `scope_accuracy_strict` + `rewrite_generated_count`: tính lại trực tiếp từ `raw_turn_results`.
- `scope_accuracy` (post-merger 2024 ambiguity tolerance trên 28 collision cores) và `rewrite_faithfulness_pct` / `rewrite_self_contained_pct`: load từ `summary_rewrite_quality.json` đã được chấm với resolver tolerance + G-Eval Judge tại thời điểm thực nghiệm.
- Judge G-Eval gốc thang 1-5; chuyển sang % bằng cách nhân 20 (chia 5 nhân 100) để đồng nhất với báo cáo.

In [10]:
from collections import defaultdict

FROZEN_SUMMARY_PATH = 'results/summary_rewrite_quality.json'

def aggregate(turn_results, frozen_summary_path=FROZEN_SUMMARY_PATH):
    with open(frozen_summary_path, 'r', encoding='utf-8') as f:
        frozen = json.load(f)

    by_pattern = defaultdict(lambda: {'total': 0, 'intent_ok': 0, 'scope_strict_ok': 0, 'rewrite_n': 0})
    for t in turn_results:
        b = by_pattern[t.get('pattern', 'unknown')]
        b['total'] += 1
        if t['predicted_intent'] == t['expected_intent']:
            b['intent_ok'] += 1
        if t['predicted_scope'] == t['expected_scope']:
            b['scope_strict_ok'] += 1
        if t.get('predicted_rewrite'):
            b['rewrite_n'] += 1

    def to_pct(score_1_5):
        return round(score_1_5 / 5 * 100, 1) if score_1_5 is not None else None

    summary = {'by_pattern': {}, 'overall': {}}
    O = {'total': 0, 'intent_ok': 0, 'scope_strict_ok': 0, 'rewrite_n': 0}

    for pat, b in by_pattern.items():
        frozen_pat = frozen['by_pattern'].get(pat, {})
        n = b['total']
        summary['by_pattern'][pat] = {
            'total_turns':                n,
            'intent_accuracy':            round(100 * b['intent_ok'] / n, 1),
            'scope_accuracy':             frozen_pat.get('scope_accuracy'),
            'scope_accuracy_strict':      round(100 * b['scope_strict_ok'] / n, 1),
            'rewrite_faithfulness_pct':   to_pct(frozen_pat.get('rewrite_faithfulness_avg')),
            'rewrite_self_contained_pct': to_pct(frozen_pat.get('rewrite_self_contained_avg')),
            'rewrite_generated_count':    b['rewrite_n'],
        }
        for k in O: O[k] += b[k]

    T = O['total']
    frozen_overall = frozen['overall']
    summary['overall'] = {
        'total_turns':                T,
        'intent_accuracy':            round(100 * O['intent_ok'] / T, 1),
        'scope_accuracy':             frozen_overall.get('scope_accuracy'),
        'scope_accuracy_strict':      round(100 * O['scope_strict_ok'] / T, 1),
        'rewrite_faithfulness_pct':   to_pct(frozen_overall.get('rewrite_faithfulness_avg')),
        'rewrite_self_contained_pct': to_pct(frozen_overall.get('rewrite_self_contained_avg')),
        'rewrite_generated_count':    O['rewrite_n'],
    }
    return summary

# Offline display: reload raw_turn_results from disk if not in memory
try:
    _ = raw_turn_results
except NameError:
    with open(RAW_TURNS_PATH, 'r', encoding='utf-8') as f:
        raw_turn_results = [json.loads(line) for line in f if line.strip()]

summary = aggregate(raw_turn_results)
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "by_pattern": {
    "location_carry_over": {
      "total_turns": 50,
      "intent_accuracy": 84.0,
      "scope_accuracy": 84.0,
      "scope_accuracy_strict": 68.0,
      "rewrite_faithfulness_pct": 97.8,
      "rewrite_self_contained_pct": 93.2,
      "rewrite_generated_count": 38
    },
    "time_shift": {
      "total_turns": 46,
      "intent_accuracy": 76.1,
      "scope_accuracy": 82.6,
      "scope_accuracy_strict": 39.1,
      "rewrite_faithfulness_pct": 94.2,
      "rewrite_self_contained_pct": 97.4,
      "rewrite_generated_count": 31
    },
    "topic_drilldown": {
      "total_turns": 46,
      "intent_accuracy": 76.1,
      "scope_accuracy": 93.5,
      "scope_accuracy_strict": 76.1,
      "rewrite_faithfulness_pct": 92.4,
      "rewrite_self_contained_pct": 86.0,
      "rewrite_generated_count": 37
    },
    "location_switch": {
      "total_turns": 46,
      "intent_accuracy": 73.9,
      "scope_accuracy": 69.6,
      "scope_accuracy_strict": 60.9,
      "rewrite

## 11. Save output

In [ ]:
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'Saved summary to {OUTPUT_PATH}')
print(f'Saved raw turns to {RAW_TURNS_PATH}')
print()
print('Download both files về máy local để cập nhật KLTN.')
